In [ ]:

"""
Chimanimani InSAR — Stage 1-2: Remote acquisition search, baseline pair
selection, and HyP3 interferogram submission.

No SLC files are downloaded to this machine at any point. HyP3 processes
server-side on ASF infrastructure using granule IDs only. Only a small,
tightly-connected subset of pairs is submitted and downloaded (see
MAX_PAIRS below) — not every candidate pair within threshold.


CREDENTIALS:
    Free Earthdata Login account: https://urs.earthdata.nasa.gov
    You'll be prompted once, when the script reaches HyP3 submission —
    hyp3_sdk handles this itself (username, then hidden password input).
    No credentials are stored in this file.
"""

import subprocess
import sys


def _ensure_installed(package):
    try:
        __import__(package)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])


for _pkg in ("asf_search", "hyp3_sdk", "shapely", "geopandas"):
    _ensure_installed(_pkg)

from collections import Counter
from datetime import datetime
from pathlib import Path
import tempfile
import zipfile

import asf_search as asf
import geopandas as gpd
import hyp3_sdk as sdk
from shapely import wkt as shapely_wkt
from shapely.geometry import shape as shapely_shape

OUTPUT_DIR = Path.cwd() / "chimanimani_insar_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def parse_time(iso_str):
    """asf_search returns startTime as an ISO-8601 string — convert to a
    real datetime so dates can be subtracted and sorted."""
    return datetime.fromisoformat(iso_str.replace("Z", "+00:00"))


# ---------------------------------------------------------------------------
# STEP 1 — Search the AOI (metadata only, nothing downloaded)
# ---------------------------------------------------------------------------

aoi_wkt = "POLYGON((32.8 -19.9, 33.1 -19.9, 33.1 -19.7, 32.8 -19.7, 32.8 -19.9))"

# --- Load the real AOI from your shapefile instead of the box above -------
# In Colab, this pops an upload button: click it and pick your zipped
# shapefile (.shp/.shx/.dbf/.prj all inside one .zip). Outside Colab, it
# asks for a local path instead.
try:
    from google.colab import files
    print("Upload your AOI shapefile as a single .zip:")
    uploaded = files.upload()
    shapefile_zip_path = next(iter(uploaded))
except ImportError:
    shapefile_zip_path = input("Path to your AOI shapefile .zip (leave blank to keep the box above): ").strip()

if shapefile_zip_path:
    extract_dir = tempfile.mkdtemp()
    with zipfile.ZipFile(shapefile_zip_path) as z:
        z.extractall(extract_dir)

    shp_files = list(Path(extract_dir).glob("*.shp"))
    if not shp_files:
        raise FileNotFoundError("No .shp file found inside that zip — check it contains .shp/.shx/.dbf/.prj.")

    gdf = gpd.read_file(shp_files[0])
    if gdf.crs is not None and gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(epsg=4326)  # asf_search expects lon/lat (WGS84)

    merged_geom = gdf.union_all() if hasattr(gdf, "union_all") else gdf.unary_union
    aoi_wkt = merged_geom.wkt
    print(f"Loaded AOI from shapefile — bounds: {gdf.total_bounds}")

aoi_polygon = shapely_wkt.loads(aoi_wkt)


def aoi_overlap_fraction(scene):
    """What fraction of the AOI's area does this scene's footprint actually
    cover? 1.0 = AOI fully inside the scene. Near 0 = the scene barely
    clips a corner of the AOI, which is exactly the bug that put a
    Mozambique-centered frame into the pipeline last time — geo_search's
    intersectsWith only guarantees SOME overlap, not meaningful coverage."""
    footprint = shapely_shape(scene.geometry)
    return footprint.intersection(aoi_polygon).area / aoi_polygon.area

results = asf.geo_search(
    platform=[asf.PLATFORM.SENTINEL1],
    intersectsWith=aoi_wkt,
    processingLevel=asf.PRODUCT_TYPE.SLC,
    beamMode="IW",
    polarization="VV+VH",
    start=datetime(2025, 9, 17),
    end=datetime(2026, 9, 17),
)

# Sentinel-1A flew with a thruster anomaly from Apr 2024 onward and was
# fully retired 29 Jun 2026, once Sentinel-1C+1D reached final orbital
# configuration and restored the nominal 6-day repeat cycle on 24 Jun 2026
# (sentinels.copernicus.eu, "Sentinel-1C/1D Final Orbital Configuration
# Achieved"). 1C alone has flown since Jan 2025, 1D joined Apr 2026 — so
# keeping only these two still covers the full window below with no gap.
results = [r for r in results if r.properties["platform"] in ("Sentinel-1C", "Sentinel-1D")]

tracks = sorted(set(r.properties["pathNumber"] for r in results))
print("Available relative orbit tracks (1C/1D only):", tracks)

# ---------------------------------------------------------------------------
# STEP 2 — Auto-select the most SBAS-consistent track
# ---------------------------------------------------------------------------
TRACK_OVERRIDE = None  # set a track number to force it; leave None for auto-select


def score_track(scenes):
    if len(scenes) < 4:
        return -1, 0.0  # too few to form a usable SBAS network

    avg_overlap = sum(aoi_overlap_fraction(s) for s in scenes) / len(scenes)

    combo_counts = Counter(
        (s.properties["beamModeType"], s.properties["polarization"], s.properties["flightDirection"])
        for s in scenes
    )
    dominant_combo, _ = combo_counts.most_common(1)[0]
    uniform = sorted(
        (s for s in scenes if (
            s.properties["beamModeType"], s.properties["polarization"], s.properties["flightDirection"]
        ) == dominant_combo),
        key=lambda s: parse_time(s.properties["startTime"]),
    )

    dates = [parse_time(s.properties["startTime"]) for s in uniform]
    gaps = [(dates[i + 1] - dates[i]).days for i in range(len(dates) - 1)]
    avg_gap = sum(gaps) / len(gaps) if gaps else 999
    longest_gap = max(gaps) if gaps else 0
    penalty = abs(avg_gap - 12) + longest_gap / 10

    base_score = len(uniform) - penalty
    # Overlap is a hard multiplier, not just another factor: a track that
    # barely clips the AOI (avg_overlap near 0) gets crushed toward zero
    # no matter how many well-spaced acquisitions it has. This is the
    # actual fix — count/regularity alone is what picked a Mozambique-
    # centered frame last time.
    return base_score * avg_overlap, avg_overlap


if TRACK_OVERRIDE is not None:
    TRACK = TRACK_OVERRIDE
    print(f"Using manually overridden track: {TRACK}")
else:
    scores = {t: score_track([r for r in results if r.properties["pathNumber"] == t]) for t in tracks}
    for t, (s, overlap) in sorted(scores.items(), key=lambda kv: kv[1][0], reverse=True):
        print(f"  track {t}: score {s:.1f}, AOI overlap {overlap * 100:.0f}%")
    TRACK = max(scores, key=lambda t: scores[t][0])
    best_overlap = scores[TRACK][1]
    print(f"Auto-selected track: {TRACK} (AOI overlap {best_overlap * 100:.0f}%)")
    if best_overlap < 0.7:
        print(
            f"WARNING: even the best track only covers {best_overlap * 100:.0f}% of the AOI. "
            "None of the available tracks may actually center on Chimanimani — verify manually "
            "in Vertex before proceeding, or supply your own shapefile-derived AOI."
        )

same_track = [r for r in results if r.properties["pathNumber"] == TRACK]

combo_counts = Counter(
    (r.properties["beamModeType"], r.properties["polarization"], r.properties["flightDirection"])
    for r in same_track
)
dominant_combo = combo_counts.most_common(1)[0][0]
if len(combo_counts) > 1:
    print(f"Mixed beam mode / polarization / flight direction on track {TRACK}: {dict(combo_counts)}")
    print(f"Keeping only: {dominant_combo}")
same_track = [
    r for r in same_track
    if (r.properties["beamModeType"], r.properties["polarization"], r.properties["flightDirection"]) == dominant_combo
]
same_track.sort(key=lambda r: parse_time(r.properties["startTime"]))

print(f"{len(same_track)} consistent acquisitions on track {TRACK}")
final_overlap = sum(aoi_overlap_fraction(r) for r in same_track) / len(same_track)
print(f"Confirmed: these acquisitions cover {final_overlap * 100:.0f}% of the AOI on average.")

# ---------------------------------------------------------------------------
# STEP 3 — Build the baseline stack, then keep only the MAX_PAIRS tightest
# small-baseline pairs (smallest combined temporal + perpendicular gap) —
# not every pair within threshold.
# ---------------------------------------------------------------------------
reference = same_track[0]

# BUGFIX: reference.stack() ignores every filter applied above — it
# re-queries ASF's full archive for this track and returns every scene
# ever acquired there, regardless of date range, platform, beam mode,
# polarization, or flight direction. Left unfiltered, this silently
# reintroduces old Sentinel-1A scenes and mismatched geometry that the
# consistency checks above were specifically built to exclude. Restrict
# it back down to only the acquisitions that actually passed those checks.
same_track_scene_names = {r.properties["sceneName"] for r in same_track}
stack = [s for s in reference.stack() if s.properties["sceneName"] in same_track_scene_names]
print(f"Baseline stack restricted to {len(stack)} of {len(same_track)} consistent acquisitions "
      f"(reference.stack() returned more before filtering).")

TEMP_BASELINE_MAX = 48    # days — upper limit for a pair to even be a candidate
PERP_BASELINE_MAX = 200   # metres — upper limit for a pair to even be a candidate
MAX_PAIRS = 25            # <-- hard cap: only this many, tightest-baseline pairs get submitted

candidates = []
for i, ref in enumerate(stack):
    for sec in stack[i + 1:]:
        dt = abs((parse_time(sec.properties["startTime"]) - parse_time(ref.properties["startTime"])).days)
        ref_b = ref.properties.get("perpendicularBaseline") or 0.0
        sec_b = sec.properties.get("perpendicularBaseline") or 0.0
        db = abs(sec_b - ref_b)
        if dt <= TEMP_BASELINE_MAX and db <= PERP_BASELINE_MAX:
            # Normalized closeness score: 0 = identical time+position, larger = further apart.
            # Combining both axes this way keeps the selection from favouring pairs that are
            # tight on one baseline but loose on the other.
            closeness = (dt / TEMP_BASELINE_MAX) + (db / PERP_BASELINE_MAX)
            candidates.append((closeness, dt, db, ref.properties["sceneName"], sec.properties["sceneName"]))

candidates.sort(key=lambda c: c[0])  # tightest first
selected = candidates[:MAX_PAIRS]
pairs = [(ref_id, sec_id) for _, _, _, ref_id, sec_id in selected]

# --- Skip pairs already submitted under this job name ----------------------
# Logging in here (rather than at the very end) so we can check what's
# already running on HyP3 before deciding what's left to submit.
hyp3 = sdk.HyP3(prompt=True)  # prompts once for Earthdata username + hidden password

remaining_credits = hyp3.check_credits()
print(f"Actual remaining HyP3 credits (from server): {remaining_credits}")

existing_jobs = hyp3.find_jobs(name="chimanimani-batch")
already_submitted = {
    tuple(sorted(job.job_parameters["granules"])) for job in existing_jobs
}
before_count = len(pairs)
pairs = [
    (ref_id, sec_id) for ref_id, sec_id in pairs
    if tuple(sorted((ref_id, sec_id))) not in already_submitted
]
skipped = before_count - len(pairs)
if skipped > 0:
    print(f"Skipping {skipped} pair(s) already submitted/running under 'chimanimani-batch'.")

print(f"{len(candidates)} candidate pairs within threshold; keeping the {len(pairs)} tightest not already submitted.")
if selected:
    worst = selected[-1]
    print(f"Loosest pair kept: {worst[1]} days apart, {worst[2]:.0f} m perpendicular baseline.")

# Connectivity check: how many of your acquisitions actually end up used?
used_scenes = set(r for pair in pairs for r in pair)
print(f"{len(used_scenes)} of {len(same_track)} acquisitions are covered by these {len(pairs)} pairs.")
unused = len(same_track) - len(used_scenes)
if unused > 0:
    print(
        f"NOTE: {unused} acquisition(s) don't appear in any selected pair and won't "
        "contribute to the time series. Raise MAX_PAIRS if you want full coverage."
    )

# Credit cost check (HyP3 Basic: 8,000 free credits/month; at 40m
# pixel spacing / 10x2 looks, a standard InSAR job costs 15 credits —
# see hyp3-docs.asf.alaska.edu/using/credits).
est_cost = len(pairs) * 15
print(f"Estimated credit cost: {est_cost} of your 8,000 monthly free credits (40m resolution).")

# ---------------------------------------------------------------------------
# STEP 4 — Submit all selected pairs as one batch, wait once
# ---------------------------------------------------------------------------
batch = sdk.Batch()
if not pairs:
    print("Nothing new to submit — every candidate pair is already running.")
else:
    for ref, sec in pairs:
        # looks="10x2" gives 40m pixel spacing — the finest resolution available
        # for whole-frame INSAR_GAMMA processing (default is "20x4" / 80m).
        # Going finer than 40m requires switching to burst-level ISCE2
        # processing (a different job type, submit_insar_isce_burst_job),
        # which needs burst-granule IDs instead of whole-scene IDs — a
        # structurally different search/pairing step, not just this parameter.
        batch += hyp3.submit_insar_job(ref, sec, name="chimanimani-batch", looks="10x2")

    batch = hyp3.watch(batch)  # one wait, for all of it
    batch.download_files(location=str(OUTPUT_DIR))
    print(f"All {len(batch)} jobs complete. Files saved to: {OUTPUT_DIR}")

Upload your AOI shapefile as a single .zip:


["'type': 'REVERSE': 'report': Reversed polygon winding order"]


Saving Chimanimani_AOI_v2.zip to Chimanimani_AOI_v2 (1).zip
Loaded AOI from shapefile — bounds: [ 32.78341316 -20.02199323  33.16578684 -19.69530907]
Available relative orbit tracks (1C/1D only): [72, 79, 174]
  track 72: score 29.9, AOI overlap 99%
  track 174: score 27.5, AOI overlap 51%
  track 79: score -1.7, AOI overlap 70%
Auto-selected track: 72 (AOI overlap 99%)
33 consistent acquisitions on track 72
Confirmed: these acquisitions cover 99% of the AOI on average.


/usr/local/lib/python3.13/dist-packages/hyp3_sdk/hyp3.py:55: UserWarning: Passing `prompt=True` is deprecated. Please use either `prompt="password"` or `prompt="token"`
  warnings.warn(


Baseline stack restricted to 33 of 33 consistent acquisitions (reference.stack() returned more before filtering).
NASA Earthdata Login username: romeothando
NASA Earthdata Login password: ··········
Actual remaining HyP3 credits (from server): 7325
Skipping 3 pair(s) already submitted/running under 'chimanimani-batch'.
129 candidate pairs within threshold; keeping the 22 tightest not already submitted.
Loosest pair kept: 13 days apart, 23 m perpendicular baseline.
28 of 33 acquisitions are covered by these 22 pairs.
NOTE: 5 acquisition(s) don't appear in any selected pair and won't contribute to the time series. Raise MAX_PAIRS if you want full coverage.
Estimated credit cost: 330 of your 8,000 monthly free credits (40m resolution).


  0%|          | 0/22 [timeout in 10800 s]

KeyboardInterrupt: 

In [ ]:
import json
from google.colab import files
from google.colab import _message

notebook_json = _message.blocking_request("get_ipynb", timeout_sec=30)["ipynb"]

with open("Data pipeline insar.ipynb", "w") as f:
    json.dump(notebook_json, f)

files.download("Data pipeline insar.ipynb")